In [ ]:
# pip install pyarrow

In [ ]:
import pandas as pd

url = "https://drive.google.com/uc?id=1D_SOblFtd-P_IfH2YUWHOFR4qSlLZlx3"
url_lookup = "https://drive.google.com/uc?id=1G3H4P3efDQQOKj5vK6Rt7FZBAvkwmAe3"

In [ ]:
df = pd.read_parquet(url)
df.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,1,2025-01-01 00:18:38,2025-01-01 00:26:59,1.0,1.60,1.0,N,229,237,1,10.0,3.5,0.5,3.00,0.0,1.0,18.00,2.5,0.0,0.0
1,1,2025-01-01 00:32:40,2025-01-01 00:35:13,1.0,0.50,1.0,N,236,237,1,5.1,3.5,0.5,2.02,0.0,1.0,12.12,2.5,0.0,0.0
2,1,2025-01-01 00:44:04,2025-01-01 00:46:01,1.0,0.60,1.0,N,141,141,1,5.1,3.5,0.5,2.00,0.0,1.0,12.10,2.5,0.0,0.0
3,2,2025-01-01 00:14:27,2025-01-01 00:20:01,3.0,0.52,1.0,N,244,244,2,7.2,1.0,0.5,0.00,0.0,1.0,9.70,0.0,0.0,0.0
4,2,2025-01-01 00:21:34,2025-01-01 00:25:06,3.0,0.66,1.0,N,244,116,2,5.8,1.0,0.5,0.00,0.0,1.0,8.30,0.0,0.0,0.0


In [ ]:
lookup = pd.read_csv(url_lookup)
lookup.head()

,LocationID,Borough,Zone,service_zone
0,1,EWR,Newark Airport,EWR
1,2,Queens,Jamaica Bay,Boro Zone
2,3,Bronx,Allerton/Pelham Gardens,Boro Zone
3,4,Manhattan,Alphabet City,Yellow Zone
4,5,Staten Island,Arden Heights,Boro Zone


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3475226 entries, 0 to 3475225
Data columns (total 20 columns):
 #   Column                 Dtype         
---  ------                 -----         
 0   VendorID               int32         
 1   tpep_pickup_datetime   datetime64[us]
 2   tpep_dropoff_datetime  datetime64[us]
 3   passenger_count        float64       
 4   trip_distance          float64       
 5   RatecodeID             float64       
 6   store_and_fwd_flag     object        
 7   PULocationID           int32         
 8   DOLocationID           int32         
 9   payment_type           int64         
 10  fare_amount            float64       
 11  extra                  float64       
 12  mta_tax                float64       
 13  tip_amount             float64       
 14  tolls_amount           float64       
 15  improvement_surcharge  float64       
 16  total_amount           float64       
 17  congestion_surcharge   float64       
 18  Airport_fee           

In [ ]:
df = df.drop_duplicates().reset_index(drop=True)
df['trip_id'] = df.index + 1
df.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,...,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee,trip_id
0,1,2025-01-01 00:18:38,2025-01-01 00:26:59,1.0,1.60,1.0,N,229,237,1,...,3.5,0.5,3.00,0.0,1.0,18.00,2.5,0.0,0.0,1
1,1,2025-01-01 00:32:40,2025-01-01 00:35:13,1.0,0.50,1.0,N,236,237,1,...,3.5,0.5,2.02,0.0,1.0,12.12,2.5,0.0,0.0,2
2,1,2025-01-01 00:44:04,2025-01-01 00:46:01,1.0,0.60,1.0,N,141,141,1,...,3.5,0.5,2.00,0.0,1.0,12.10,2.5,0.0,0.0,3
3,2,2025-01-01 00:14:27,2025-01-01 00:20:01,3.0,0.52,1.0,N,244,244,2,...,1.0,0.5,0.00,0.0,1.0,9.70,0.0,0.0,0.0,4
4,2,2025-01-01 00:21:34,2025-01-01 00:25:06,3.0,0.66,1.0,N,244,116,2,...,1.0,0.5,0.00,0.0,1.0,8.30,0.0,0.0,0.0,5


In [ ]:
pickup_datetime_dim = df[['tpep_pickup_datetime']].drop_duplicates().reset_index(drop=True)
pickup_datetime_dim['hour'] = pickup_datetime_dim['tpep_pickup_datetime'].dt.hour
pickup_datetime_dim['day'] = pickup_datetime_dim['tpep_pickup_datetime'].dt.day
pickup_datetime_dim['month'] = pickup_datetime_dim['tpep_pickup_datetime'].dt.month
pickup_datetime_dim['year'] = pickup_datetime_dim['tpep_pickup_datetime'].dt.year
pickup_datetime_dim['weekday'] = pickup_datetime_dim['tpep_pickup_datetime'].dt.weekday
pickup_datetime_dim['weekday_name'] = pickup_datetime_dim['tpep_pickup_datetime'].dt.day_name()
pickup_datetime_dim['quarter'] = pickup_datetime_dim['tpep_pickup_datetime'].dt.quarter

# Create surrogate key
pickup_datetime_dim['pickup_datetime_id'] = pickup_datetime_dim.index + 1

# Reorder columns (match ERD)
pickup_datetime_dim = pickup_datetime_dim[
    [
        'pickup_datetime_id',
        'tpep_pickup_datetime',
        'hour',
        'day',
        'month',
        'year',
        'weekday',
        'weekday_name',
        'quarter'
    ]
]

pickup_datetime_dim.head()

,pickup_datetime_id,tpep_pickup_datetime,hour,day,month,year,weekday,weekday_name,quarter
0,1,2025-01-01 00:18:38,0,1,1,2025,2,Wednesday,1
1,2,2025-01-01 00:32:40,0,1,1,2025,2,Wednesday,1
2,3,2025-01-01 00:44:04,0,1,1,2025,2,Wednesday,1
3,4,2025-01-01 00:14:27,0,1,1,2025,2,Wednesday,1
4,5,2025-01-01 00:21:34,0,1,1,2025,2,Wednesday,1


In [ ]:
dropoff_datetime_dim = df[['tpep_dropoff_datetime']].drop_duplicates().reset_index(drop=True)

dropoff_datetime_dim['hour'] = dropoff_datetime_dim['tpep_dropoff_datetime'].dt.hour
dropoff_datetime_dim['day'] = dropoff_datetime_dim['tpep_dropoff_datetime'].dt.day
dropoff_datetime_dim['month'] = dropoff_datetime_dim['tpep_dropoff_datetime'].dt.month
dropoff_datetime_dim['year'] = dropoff_datetime_dim['tpep_dropoff_datetime'].dt.year
dropoff_datetime_dim['weekday'] = dropoff_datetime_dim['tpep_dropoff_datetime'].dt.weekday
dropoff_datetime_dim['weekday_name'] = dropoff_datetime_dim['tpep_dropoff_datetime'].dt.day_name()
dropoff_datetime_dim['quarter'] = dropoff_datetime_dim['tpep_dropoff_datetime'].dt.quarter

# Create surrogate key
dropoff_datetime_dim['dropoff_datetime_id'] = dropoff_datetime_dim.index + 1

# Reorder columns (match ERD)
dropoff_datetime_dim = dropoff_datetime_dim[
    [
        'dropoff_datetime_id',
        'tpep_dropoff_datetime',
        'hour',
        'day',
        'month',
        'year',
        'weekday',
        'weekday_name',
        'quarter'
    ]
]

dropoff_datetime_dim.head()

,dropoff_datetime_id,tpep_dropoff_datetime,hour,day,month,year,weekday,weekday_name,quarter
0,1,2025-01-01 00:26:59,0,1,1,2025,2,Wednesday,1
1,2,2025-01-01 00:35:13,0,1,1,2025,2,Wednesday,1
2,3,2025-01-01 00:46:01,0,1,1,2025,2,Wednesday,1
3,4,2025-01-01 00:20:01,0,1,1,2025,2,Wednesday,1
4,5,2025-01-01 00:25:06,0,1,1,2025,2,Wednesday,1


In [ ]:
# Create passenger count dimension
passenger_count_dim = df[['passenger_count']].copy()
passenger_count_dim['passenger_count'] = passenger_count_dim['passenger_count'].fillna(0).astype(int)
passenger_count_dim = passenger_count_dim[['passenger_count']].drop_duplicates().reset_index(drop=True)
passenger_count_dim = passenger_count_dim.sort_values(by='passenger_count').reset_index(drop=True)


passenger_count_dim['passenger_count_id'] = passenger_count_dim.index + 1

passenger_count_dim = passenger_count_dim[
    [
        'passenger_count_id',
        'passenger_count'
    ]
]

passenger_count_dim.head()

,passenger_count_id,passenger_count
0,1,0
1,2,1
2,3,2
3,4,3
4,5,4


In [ ]:
trip_distance_dim = df[['trip_distance']].copy()

trip_distance_dim['trip_distance'] = (
    trip_distance_dim['trip_distance']
    .fillna(0)
    # .round(2)
)

trip_distance_dim = trip_distance_dim.drop_duplicates().reset_index(drop=True)
trip_distance_dim = trip_distance_dim.sort_values(by='trip_distance').reset_index(drop=True)

trip_distance_dim['trip_distance_id'] = trip_distance_dim.index + 1

trip_distance_dim = trip_distance_dim[['trip_distance_id', 'trip_distance']]

trip_distance_dim.head()

,trip_distance_id,trip_distance
0,1,0.00
1,2,0.01
2,3,0.02
3,4,0.03
4,5,0.04


In [ ]:
rate_code_mapping = {
    1: "Standard rate",
    2: "JFK",
    3: "Newark",
    4: "Nassau or Westchester",
    5: "Negotiated fare",
    6: "Group ride",
    99: "Unknown"
}
rate_code_dim = df[['RatecodeID']].copy()
rate_code_dim['RatecodeID'] = df['RatecodeID'].fillna(99).astype(int)
rate_code_dim = rate_code_dim.drop_duplicates().reset_index(drop=True)
rate_code_dim['rate_code_name'] = rate_code_dim['RatecodeID'].map(rate_code_mapping)
rate_code_dim = rate_code_dim.sort_values(by='RatecodeID').reset_index(drop=True)

rate_code_dim['rate_code_id'] = rate_code_dim.index + 1

rate_code_dim = rate_code_dim[
    ['rate_code_id', 'RatecodeID', 'rate_code_name']
]
rate_code_dim.head()

,rate_code_id,RatecodeID,rate_code_name
0,1,1,Standard rate
1,2,2,JFK
2,3,3,Newark
3,4,4,Nassau or Westchester
4,5,5,Negotiated fare


In [ ]:
store_and_fwd_mapping = {
    'Y': 'Stored and forwarded later',
    'N': 'Sent in real-time',
    'U': 'Unknown'
}

store_and_fwd_dim = df[['store_and_fwd_flag']].copy()
store_and_fwd_dim['store_and_fwd_flag'] = store_and_fwd_dim['store_and_fwd_flag'].fillna('U')
store_and_fwd_dim = store_and_fwd_dim.drop_duplicates().reset_index(drop=True)
store_and_fwd_dim['description'] = store_and_fwd_dim['store_and_fwd_flag'].map(store_and_fwd_mapping)
store_and_fwd_dim = store_and_fwd_dim.sort_values(by='store_and_fwd_flag').reset_index(drop=True)

store_and_fwd_dim['store_fwd_flag_id'] = store_and_fwd_dim.index + 1

store_and_fwd_dim = store_and_fwd_dim[
    ['store_fwd_flag_id', 'store_and_fwd_flag', 'description']
]
store_and_fwd_dim.head()

,store_fwd_flag_id,store_and_fwd_flag,description
0,1,N,Sent in real-time
1,2,U,Unknown
2,3,Y,Stored and forwarded later


In [ ]:
# pickup_location_dim
pickup_location_dim = df[['PULocationID']].copy()

pickup_location_dim = pickup_location_dim.drop_duplicates().reset_index(drop=True)

pickup_location_dim = pickup_location_dim.merge(
    lookup,
    left_on='PULocationID',
    right_on='LocationID',
    how='left'
)

pickup_location_dim['pickup_location_id'] = pickup_location_dim.index + 1

pickup_location_dim = pickup_location_dim[
    [
        'pickup_location_id',
        'PULocationID',
        'Borough',
        'Zone',
        'service_zone'
    ]
]

pickup_location_dim.head()

,pickup_location_id,PULocationID,Borough,Zone,service_zone
0,1,229,Manhattan,Sutton Place/Turtle Bay North,Yellow Zone
1,2,236,Manhattan,Upper East Side North,Yellow Zone
2,3,141,Manhattan,Lenox Hill West,Yellow Zone
3,4,244,Manhattan,Washington Heights South,Boro Zone
4,5,239,Manhattan,Upper West Side South,Yellow Zone


In [ ]:
# dropoff_location_dim
dropoff_location_dim = df[['DOLocationID']].copy()

dropoff_location_dim = dropoff_location_dim.drop_duplicates().reset_index(drop=True)

dropoff_location_dim = dropoff_location_dim.merge(
    lookup,
    left_on='DOLocationID',
    right_on='LocationID',
    how='left'
)

dropoff_location_dim['dropoff_location_id'] = dropoff_location_dim.index + 1

dropoff_location_dim = dropoff_location_dim[
    [
        'dropoff_location_id',
        'DOLocationID',
        'Borough',
        'Zone',
        'service_zone'
    ]
]

dropoff_location_dim.head()

,dropoff_location_id,DOLocationID,Borough,Zone,service_zone
0,1,237,Manhattan,Upper East Side South,Yellow Zone
1,2,141,Manhattan,Lenox Hill West,Yellow Zone
2,3,244,Manhattan,Washington Heights South,Boro Zone
3,4,116,Manhattan,Hamilton Heights,Boro Zone
4,5,68,Manhattan,East Chelsea,Yellow Zone


In [ ]:
payment_mapping = {
    0: "Flex Fare trip",
    1: "Credit card",
    2: "Cash",
    3: "No charge",
    4: "Dispute",
    5: "Unknown",
    6: "Voided trip"
}

payment_type_dim = df[['payment_type']].copy()
payment_type_dim['payment_type'] = payment_type_dim['payment_type'].fillna(5).astype(int)
payment_type_dim = payment_type_dim.drop_duplicates().reset_index(drop=True)
payment_type_dim['payment_type_name'] = payment_type_dim['payment_type'].map(payment_mapping)

payment_type_dim = payment_type_dim.sort_values(by='payment_type').reset_index(drop=True)

payment_type_dim['payment_type_id'] = payment_type_dim.index + 1

payment_type_dim = payment_type_dim[
    ['payment_type_id', 'payment_type', 'payment_type_name']
]
payment_type_dim.head()

,payment_type_id,payment_type,payment_type_name
0,1,0,Flex Fare trip
1,2,1,Credit card
2,3,2,Cash
3,4,3,No charge
4,5,4,Dispute


In [ ]:
fact_table = df.copy()

# Merge all dimension tables using correct keys
fact_table = fact_table.merge(passenger_count_dim, on='passenger_count', how='left') \
    .merge(trip_distance_dim, on='trip_distance', how='left') \
    .merge(rate_code_dim, on='RatecodeID', how='left') \
    .merge(pickup_location_dim, on='PULocationID', how='left') \
    .merge(dropoff_location_dim, on='DOLocationID', how='left') \
    .merge(pickup_datetime_dim, on='tpep_pickup_datetime', how='left') \
    .merge(dropoff_datetime_dim, on='tpep_dropoff_datetime', how='left') \
    .merge(payment_type_dim, on='payment_type', how='left') \
    .merge(store_and_fwd_dim, on='store_and_fwd_flag', how='left')

# Select columns as per ERD
fact_table = fact_table[
    [
        'trip_id',
        'VendorID',
        'pickup_datetime_id',
        'dropoff_datetime_id',
        'passenger_count_id',
        'trip_distance_id',
        'rate_code_id',
        'store_fwd_flag_id',
        'pickup_location_id',
        'dropoff_location_id',
        'payment_type_id',
        'fare_amount',
        'extra',
        'mta_tax',
        'tip_amount',
        'tolls_amount',
        'improvement_surcharge',
        'congestion_surcharge',
        'Airport_fee',
        'cbd_congestion_fee',
        'total_amount'
    ]
]

fact_table.head()

,trip_id,VendorID,pickup_datetime_id,dropoff_datetime_id,passenger_count_id,trip_distance_id,rate_code_id,store_fwd_flag_id,pickup_location_id,dropoff_location_id,...,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,congestion_surcharge,Airport_fee,cbd_congestion_fee,total_amount
0,1,1,1,1,2.0,161,1.0,1.0,1,1,...,10.0,3.5,0.5,3.00,0.0,1.0,2.5,0.0,0.0,18.00
1,2,1,2,2,2.0,51,1.0,1.0,2,1,...,5.1,3.5,0.5,2.02,0.0,1.0,2.5,0.0,0.0,12.12
2,3,1,3,3,2.0,61,1.0,1.0,3,2,...,5.1,3.5,0.5,2.00,0.0,1.0,2.5,0.0,0.0,12.10
3,4,2,4,4,4.0,53,1.0,1.0,4,3,...,7.2,1.0,0.5,0.00,0.0,1.0,0.0,0.0,0.0,9.70
4,5,2,5,5,4.0,67,1.0,1.0,4,4,...,5.8,1.0,0.5,0.00,0.0,1.0,0.0,0.0,0.0,8.30
